# 🤖 GirishOS Backend - Google Colab Setup

This notebook runs the GirishOS backend on Google Colab with free GPU.

## Prerequisites:
1. A photo of Girish (front-facing, neutral expression, 512x512+)
2. A free Groq API key from https://console.groq.com
3. A free ngrok auth token from https://dashboard.ngrok.com

## How it works:
1. Cell 1: Clones your GitHub repo (gets all backend code)
2. Cell 2: Installs dependencies + SadTalker
3. Cell 3: Upload your photo
4. Cell 4: Configure API keys
5. Cell 5: Start server (gives you a public URL)

Your frontend connects to the public URL via WebSocket.

In [ ]:
# ============================================
# CELL 1: Clone your GitHub repo
# ============================================
import os

REPO_URL = "https://github.com/vishwjeet-ujgare-hd/AI-Powered-Meeting-Host.git"
PROJECT_DIR = "AI-Powered-Meeting-Host"

# Clone or pull latest
if os.path.exists(PROJECT_DIR):
    print("📂 Repo already exists, pulling latest changes...")
    os.chdir(PROJECT_DIR)
    !git pull
else:
    print("📥 Cloning repo from GitHub...")
    !git clone {REPO_URL}
    os.chdir(PROJECT_DIR)

print(f"\n✅ Project loaded! Working directory: {os.getcwd()}")
print(f"📁 Files: {os.listdir('.')}")

In [ ]:
# ============================================
# CELL 2: Install dependencies + SadTalker
# ============================================

# Install Python packages
!pip install -q fastapi uvicorn[standard] edge-tts groq pyngrok nest-asyncio websockets python-multipart aiofiles

# Clone and setup SadTalker (face animation engine)
import os
if not os.path.exists("SadTalker"):
    print("\n🎭 Setting up SadTalker (face animation)...")
    !git clone https://github.com/OpenTalker/SadTalker.git
    %cd SadTalker
    !pip install -q -r requirements.txt
    !bash scripts/download_models.sh
    %cd ..
else:
    print("🎭 SadTalker already set up")

print("\n✅ All dependencies installed!")

In [ ]:
# ============================================
# CELL 3: Upload Girish's photo
# ============================================
import os
from google.colab import files

os.makedirs("assets/photos", exist_ok=True)

# Check if photo already exists
existing_photos = [f for f in os.listdir("assets/photos") if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if existing_photos:
    print(f"📸 Photo already exists: assets/photos/{existing_photos[0]}")
    print("   (Run this cell again if you want to upload a new one)")
else:
    print("📸 Upload Girish's photo")
    print("   Requirements:")
    print("   - PNG or JPG format")
    print("   - Front-facing, looking at camera")
    print("   - Neutral expression (slight smile OK)")
    print("   - Good lighting, plain background")
    print("   - At least 512x512 pixels")
    print("")
    
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        dest = f"assets/photos/{filename}"
        os.rename(filename, dest)
        print(f"\n✅ Photo saved as: {dest}")

In [ ]:
# ============================================
# CELL 4: Configure API keys
# ============================================
# 
# SETUP: Go to left panel → 🔑 (Secrets) → Add:
#   - GROQ_API_KEY     (from https://console.groq.com)
#   - NGROK_AUTH_TOKEN  (from https://dashboard.ngrok.com)
# Then toggle "Notebook access" ON for both.
#
import os
from google.colab import userdata

# Load Groq API key
try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    print("✅ Groq API key loaded from Colab Secrets")
except Exception:
    GROQ_API_KEY = input("Enter your Groq API key: ")

# Load ngrok auth token
try:
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
    print("✅ ngrok auth token loaded from Colab Secrets")
except Exception:
    NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ")

# Set as environment variables (backend code reads from these)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN

# Find the photo
photo_path = None
for f in os.listdir("assets/photos"):
    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
        photo_path = f"assets/photos/{f}"
        break

if photo_path:
    print(f"✅ Using photo: {photo_path}")
    os.environ["AVATAR_PHOTO_PATH"] = photo_path
else:
    print("❌ No photo found! Run Cell 3 first.")

print("\n🎯 Configuration complete!")

In [ ]:
# ============================================
# CELL 5: Start the GirishOS backend server
# ============================================
# ⚠️ Keep this cell running during the demo!
#
import sys
import nest_asyncio
nest_asyncio.apply()

# Add backend to Python path
sys.path.insert(0, "backend")

from pyngrok import ngrok

# Start ngrok tunnel
ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])
public_url = ngrok.connect(8000, "http")
ws_url = str(public_url).replace("https://", "wss://").replace("http://", "ws://")

print("=" * 60)
print("🚀 GirishOS Backend is LIVE!")
print("=" * 60)
print(f"")
print(f"📡 HTTP URL:      {public_url}")
print(f"🔌 WebSocket URL: {ws_url}/ws")
print(f"")
print(f"📋 Copy this to your frontend .env.local file:")
print(f"   NEXT_PUBLIC_BACKEND_URL={ws_url}")
print(f"")
print(f"🧪 Test it: Open {public_url}/health in browser")
print("=" * 60)
print("")
print("⚠️  Keep this cell running! Don't stop it during demo.")
print("")

# Start FastAPI server
import uvicorn
from main import app
uvicorn.run(app, host="0.0.0.0", port=8000)